# adg-M4: silence shortcut and frequency bands (GPU part)

Settings: **Accelerator = GPU (T4 or P100)**, **Internet = on**, **Input = dataset `awsaf49/asvpoof-2019-dataset`**. Run with *Save Version, Save & Run All*.

1. checks the ASVspoof copy against my official download, downloads In-the-Wild;
2. LFCC frames: train/dev/eval/In-the-Wild as they are, the same with leading/trailing silence trimmed (`trim40`), and the stratified eval subset (`results/m4/band_subset_eval.csv`, 5,900 files) with each 1 kHz band removed;
3. wav2vec 2.0 features for the trimmed sets and the band-masked subsets (the untouched ones exist from milestones 2-3);
4. LFCC + LCNN trained as in milestone 3 (weights kept this time), scored on everything above;
5. LFCC + LCNN trained and selected on trimmed audio, scored on trimmed eval and In-the-Wild;
6. the milestone 1 GMMs scored on the band-masked subsets and on trimmed In-the-Wild.

Every step goes through `run()`, which stops the notebook at the first error.

Output in `/kaggle/working`: `w2v2_m4/` (about 7.5 GB: 5.6 GB trimmed sets + 1.7 GB band subsets) and `results/m4/` (score files).

In [ ]:
import os, shutil, subprocess, torch, torchaudio
print('torch', torch.__version__, '| torchaudio', torchaudio.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable the GPU accelerator')
print('CPU cores:', os.cpu_count(), '| free disk in /tmp: %.0f GB' % (shutil.disk_usage('/tmp').free / 1e9))

def run(cmd):
    """Run a shell command, stream its output, raise if it fails."""
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

BANDS = [f'band:{lo}-{lo + 1000}' for lo in range(0, 8000, 1000)]
SUBSET = 'results/m4/band_subset_eval.csv'
FEAT = '/tmp/features'
LFCC = f'{FEAT}/lfcc_asvspoof2019_b02'

In [ ]:
run('rm -rf /tmp/repo && git clone --depth 1 https://github.com/amine-belkacem/audio-deepfake-generalization.git /tmp/repo')
run('pip install -q soundfile')
os.chdir('/tmp/repo')

In [ ]:
LA = subprocess.check_output('python scripts/verify_asvspoof_copy.py --find /kaggle/input', shell=True, text=True).strip()
print('ASVspoof 2019 LA copy found in', LA)
run(f'python scripts/verify_asvspoof_copy.py --la-root "{LA}"')
run('python scripts/download_in_the_wild.py --out /tmp/itw')
COMMON = f'--la-root "{LA}" --itw-root /tmp/itw'

In [ ]:
# LFCC frames (CPU): untouched, trimmed, and band-masked subsets
run(f'python scripts/extract_lfcc.py {COMMON} --feat-root {FEAT} --partitions train dev eval itw')
run(f'python scripts/extract_lfcc.py {COMMON} --feat-root {FEAT} --partitions train dev eval itw --transform trim40')
run(f'python scripts/extract_lfcc.py {COMMON} --feat-root {FEAT} --partitions eval --subset {SUBSET}')
for b in BANDS:
    run(f'python scripts/extract_lfcc.py {COMMON} --feat-root {FEAT} --partitions eval --subset {SUBSET} --transform {b}')

In [ ]:
# wav2vec 2.0 (GPU): trimmed sets and band-masked subsets
W2V = '/kaggle/working/w2v2_m4'
run(f'python scripts/extract_wav2vec2.py {COMMON} --out {W2V} --device cuda --partitions train dev eval itw --transform trim40')
for b in BANDS:
    run(f'python scripts/extract_wav2vec2.py {COMMON} --out {W2V} --device cuda --partitions eval --subset {SUBSET} --transform {b}')

In [ ]:
# LCNN trained as in milestone 3, scored on untouched, trimmed and band-masked data
parts = 'eval itw eval_trim40 itw_trim40 evalsub ' + ' '.join('evalsub_' + b.replace(':', '') for b in BANDS)
run(f'python scripts/train_lcnn.py --feat-dir {LFCC} {COMMON} --out /kaggle/working/results/m4/lfcc_lcnn_normal '
    f'--device cuda --eval-parts {parts}')

In [ ]:
# LCNN trained and selected on trimmed audio
run(f'python scripts/train_lcnn.py --feat-dir {LFCC} {COMMON} --out /kaggle/working/results/m4/lfcc_lcnn_trim40 '
    '--device cuda --train-part train_trim40 --dev-part dev_trim40 --eval-parts eval_trim40 itw_trim40')

In [ ]:
# milestone 1 GMMs on the band-masked subsets and on trimmed In-the-Wild
OUT = '/kaggle/working/results/m4/lfcc_gmm'
for part in ['evalsub', 'itw_trim40'] + ['evalsub_' + b.replace(':', '') for b in BANDS]:
    run(f'python scripts/score_gmm.py --gmm-dir results/lfcc_gmm_asvspoof2019_b02 --feat-dir {LFCC}/{part} --out {OUT}/scores_{part}.csv')

In [ ]:
run('du -sh /kaggle/working/*')
run('ls -la /kaggle/working/w2v2_m4 /kaggle/working/results/m4/*')